<a href="https://colab.research.google.com/github/mashan275/data-science-2024/blob/main/Pertemuan6_Hasta_satriya_240401010207.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#langkah 0  Import library yang dibutuhkan
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Atur tampilan grafik
plt.style.use('seaborn-v0_8')


#=================================##======================================#
#Langkah 1: Load Data & Analisis Awal (EDA)                               #
# Memuat data, mengecek ukuran data, nilai kosong, dan distribusi target. #
#=================================##======================================#
df = sns.load_dataset('titanic')

# Pilih kolom yang akan digunakan sesuai panduan
cols = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'survived']
df = df[cols].copy()  # Membuat salinan agar data asli tidak berubah

# 1. Cek ukuran data
print("=== Ukuran Dataset ===")
print(f"Jumlah Baris: {df.shape[0]}, Jumlah Kolom: {df.shape[1]}")

# 2. Cek nilai yang hilang (Missing Values)
print("\n=== Jumlah Nilai Hilang di Setiap Kolom ===")
print(df.isnull().sum())

# 3. Cek distribusi kolom target ('survived')
print("\n=== Distribusi Target (Survived) ===")
print(df['survived'].value_counts(normalize=True).round(3) * 100)
#====================================##====================================#



#=================================##======================================#
#Langkah 2: Penanganan Nilai Hilang (Handling Missing Values)             #
#Mengisi data kosong sesuai aturan: Median untuk data numerik,            #
#Modus untuk data kategori.                                               #
#=================================##======================================#
# Isi kolom 'age' dengan MEDIAN (tahan terhadap nilai ekstrem/outlier)
df['age'] = df['age'].fillna(df['age'].median())

# Isi kolom 'embarked' dengan MODUS (nilai yang paling sering muncul)
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# Cek ulang untuk memastikan sudah tidak ada nilai kosong
print("=== Cek Ulang Nilai Hilang (Setelah Penanganan) ===")
print(df.isnull().sum())
#====================================##====================================#


#=================================##===========================================#
#Langkah 3: Encoding Data Kategorikal                                          #
#Mengubah teks menjadi angka menggunakan One-Hot Encoding agar bisa diproses   #
#komputer. Kita gunakan drop_first=True untuk menghindari Dummy Variable Trap. #
#=================================##===========================================#
# Melakukan One-Hot Encoding pada kolom 'sex' dan 'embarked'
df = pd.get_dummies(
    df,
    columns=['sex', 'embarked'],   # Kolom yang mau diubah
    drop_first=True,               # Hapus satu kolom untuk mencegah keterkaitan data
    dtype=int                      # Hasilnya berupa angka 0/1, bukan True/False
)

# Lihat nama kolom baru setelah proses encoding
print("=== Nama Kolom Setelah Encoding ===")
print(df.columns.tolist())
#=================================##===========================================#


#=================================##===========================================#
#Langkah 4: Membagi Data (Train-Test Split)                                    #
#Memisahkan data menjadi data latih (80%) dan data uji (20%).                  #
#Wajib gunakan stratify=y karena data target tidak seimbang.                   #
#=================================##===========================================#
from sklearn.model_selection import train_test_split

# Pisahkan Fitur (X) dan Target (y)
X = df.drop('survived', axis=1)  # Semua kolom KECUALI 'survived'
y = df['survived']                # Kolom jawaban/target

# Lakukan pembagian data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% untuk data uji
    random_state=42,     # Agar hasil acak selalu sama setiap kali dijalankan
    stratify=y            # Menjaga keseimbangan persentase selamat/tidak selamat
)

# Cek ukuran pembagian
print(f"Data Latih: {X_train.shape[0]} baris")
print(f"Data Uji : {X_test.shape[0]} baris")

# Verifikasi proporsi kelas agar seimbang antara latih & uji
print("\n=== Proporsi Kelas di Data Latih ===")
print(y_train.value_counts(normalize=True).round(3))
print("\n=== Proporsi Kelas di Data Uji ===")
print(y_test.value_counts(normalize=True).round(3))
#=================================##===========================================#


#=================================##===========================================#
#Langkah 5: Penskalaan Fitur (Feature Scaling)                                 #
#Mengubah rentang nilai kolom angka agar seragam  StandardScaler               #
#Kolom hasil One-Hot Encoding (0/1) TIDAK PERLU di-skala.                      #
#=================================##===========================================#
from sklearn.preprocessing import StandardScaler

# Tentukan kolom numerik yang perlu di-skala
num_cols = ['pclass', 'age', 'sibsp', 'parch', 'fare']

# Inisialisasi Scaler
scaler = StandardScaler()

# ===== ATURAN UTAMA =====
# 1. Pasang & Belajar pola HANYA dari Data Latih
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# 2. Terapkan pola tersebut ke Data Uji (cukup .transform saja, jangan fit ulang!)
X_test[num_cols] = scaler.transform(X_test[num_cols])
# ========================

# Tampilkan hasil akhir
print("=== Rata-rata asli dari data latih ===")
print(scaler.mean_.round(2))

print("\n=== Contoh Data Latih Setelah Penskalaan ===")
print(X_train.head().round(3))

print("\n✅ Data sudah bersih dan siap digunakan untuk model Machine Learning!")
print(f"Ukuran X_train: {X_train.shape}, Ukuran y_train: {y_train.shape}")
print(f"Ukuran X_test : {X_test.shape}, Ukuran y_test : {y_test.shape}")

=== Ukuran Dataset ===
Jumlah Baris: 891, Jumlah Kolom: 8

=== Jumlah Nilai Hilang di Setiap Kolom ===
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
survived      0
dtype: int64

=== Distribusi Target (Survived) ===
survived
0    61.6
1    38.4
Name: proportion, dtype: float64
=== Cek Ulang Nilai Hilang (Setelah Penanganan) ===
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
survived    0
dtype: int64
=== Nama Kolom Setelah Encoding ===
['pclass', 'age', 'sibsp', 'parch', 'fare', 'survived', 'sex_male', 'embarked_Q', 'embarked_S']
Data Latih: 712 baris
Data Uji : 179 baris

=== Proporsi Kelas di Data Latih ===
survived
0    0.617
1    0.383
Name: proportion, dtype: float64

=== Proporsi Kelas di Data Uji ===
survived
0    0.615
1    0.385
Name: proportion, dtype: float64
=== Rata-rata asli dari data latih ===
[ 2.31 29.46  0.49  0.39 31.82]

=== Contoh Data Latih Setelah 